# 01 - Load and explore the IndPenSim batch data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AHichem/resilient-process-monitoring/blob/main/notebooks/01_load_and_explore.ipynb)

**Goal of this notebook:** download the IndPenSim dataset, keep only the process variables, save a compact copy, and take a first look at the 100 batches.

**About the data.** IndPenSim simulates a 100,000 L industrial penicillin fermentation (Goldrick et al., 2015 and 2019). It contains 100 fed-batch runs operated with different control strategies:

| Batches | Control strategy |
|---|---|
| 1-30 | Recipe-driven |
| 31-60 | Operator-controlled |
| 61-90 | Advanced Process Control (APC) using Raman |
| 91-100 | Contain process faults |

Source: [Mendeley Data, DOI 10.17632/pdnjz7zz5x](https://data.mendeley.com/datasets/pdnjz7zz5x), licence CC BY 4.0.

## 1. Setup

Mounting Google Drive is optional but recommended: the cleaned file is cached there, so you only download the 0.5 GB zip once.

In [ ]:
USE_DRIVE = True  # set to False if you do not want to use Google Drive

from pathlib import Path

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DATA_DIR = Path('/content/drive/MyDrive/indpensim')
    except ImportError:  # not running in Colab
        DATA_DIR = Path('data')
else:
    DATA_DIR = Path('data')

DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DATA_DIR / '100_Batches_IndPenSim.zip'
RAW_CSV = DATA_DIR / 'Mendeley_data' / '100_Batches_IndPenSim_V3.csv'
PROCESS_PARQUET = DATA_DIR / 'indpensim_process.parquet'
print('Data folder:', DATA_DIR.resolve())

## 2. Download the data (only the first time)

The link below is the one used by the dataset authors in their own notebook. The download takes a few minutes.

In [ ]:
import zipfile
import requests

URL = ('https://data.mendeley.com/datasets/pdnjz7zz5x/1/files/'
       'ec0dfb55-7e3c-4124-8b0c-ca8ae7c50fe7/100_Batches_IndPenSim.zip')

if PROCESS_PARQUET.exists():
    print('Clean file already cached - skipping download.')
else:
    if not ZIP_PATH.exists():
        print('Downloading... (about 0.5 GB)')
        with requests.get(URL, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(ZIP_PATH, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
    if not RAW_CSV.exists():
        print('Unzipping...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall(DATA_DIR)
    print('Raw CSV ready:', RAW_CSV.exists())

## 3. Keep only the process variables

The raw CSV has about 40 process variables followed by about 2,200 Raman spectrum columns (named by wavenumber, starting at `2400`). For process monitoring we start with the process variables only, which keeps memory use small.

**Known quirk:** in the raw file, the labels of the columns *Batch reference* and *2-PAT control* are swapped. The dataset authors fix this in their own notebook, and we do the same.

In [ ]:
import numpy as np
import pandas as pd

def load_process_data(raw_csv):
    """Read only the process-variable columns of the IndPenSim CSV."""
    header = pd.read_csv(raw_csv, nrows=0).columns
    first_raman = header.get_loc('2400')
    process_cols = list(header[:first_raman])
    df = pd.read_csv(raw_csv, usecols=process_cols)
    # Fix the swapped labels (same fix as the dataset authors)
    df = df.rename(columns={
        '2-PAT control(PAT_ref:PAT ref)': 'Batch reference(Batch_ref:Batch ref)',
        'Batch reference(Batch_ref:Batch ref)': '2-PAT control(PAT_ref:PAT ref)',
    })
    df = df.rename(columns={'Batch reference(Batch_ref:Batch ref)': 'batch_id'})
    df['batch_id'] = df['batch_id'].astype(int)
    float_cols = df.select_dtypes('float64').columns
    df[float_cols] = df[float_cols].astype('float32')
    return df

def control_group(batch_id):
    if batch_id <= 30:
        return 'recipe'
    if batch_id <= 60:
        return 'operator'
    if batch_id <= 90:
        return 'APC (Raman)'
    return 'fault'

if PROCESS_PARQUET.exists():
    df = pd.read_parquet(PROCESS_PARQUET)
else:
    df = load_process_data(RAW_CSV)
    df['group'] = df['batch_id'].map(control_group)
    df.to_parquet(PROCESS_PARQUET, index=False)
    print('Saved clean copy to', PROCESS_PARQUET)

print(df.shape)
df.head()

## 4. Sanity checks

Three questions to answer before any modelling:
1. Do we really have 100 batches, and how long is each one?
2. Which variables are measured online (every time step) and which offline (lab samples, mostly empty)?
3. Do the batch groups look as described?

In [ ]:
print('Number of batches:', df['batch_id'].nunique())

batch_length = df.groupby('batch_id')['Time (h)'].max()
print('\nBatch duration (h):')
print(batch_length.describe().round(1))

In [ ]:
# Share of missing values per column: offline measurements are sampled
# only every few hours, so many NaN there are expected, not an error.
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0].round(3)

In [ ]:
df.groupby('group')['batch_id'].nunique()

## 5. First look: penicillin concentration over time

Each line is one batch, coloured by control strategy. Look for differences in the final titre between groups, and for batches that behave differently.

In [ ]:
import matplotlib.pyplot as plt

colors = {'recipe': 'tab:blue', 'operator': 'tab:orange',
          'APC (Raman)': 'tab:green', 'fault': 'tab:red'}
y_col = 'Penicillin concentration(P:g/L)'

fig, ax = plt.subplots(figsize=(10, 5))
for (batch, group), g in df.groupby(['batch_id', 'group']):
    ax.plot(g['Time (h)'], g[y_col], color=colors[group], alpha=0.5, lw=0.8)
for group, c in colors.items():
    ax.plot([], [], color=c, label=group)
ax.set_xlabel('Time (h)')
ax.set_ylabel('Penicillin (g/L)')
ax.set_title('IndPenSim - penicillin concentration, 100 batches')
ax.legend(title='Control strategy')
plt.tight_layout()
plt.show()

## 6. Your turn

- [ ] Print `df.columns` and write, for each variable, whether it is a manipulated input, an online sensor or an offline lab measurement.
- [ ] Repeat the plot above for temperature, pH and dissolved oxygen.
- [ ] Compute the final penicillin concentration per batch and compare the four groups with a box plot.
- [ ] Plot batch 91 against a normal batch and describe what goes wrong.

Next notebook: build a "golden batch" from normal batches and a first monitoring chart.